### Executar este notebook no Jupyter através do navegador

**URL:** *localhost:8888*  
**Senha:** *1234*

In [ ]:
!pip install dotenv boto3


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import sys
from dotenv import load_dotenv

import boto3
from botocore.exceptions import ClientError

from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, from_unixtime
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, BooleanType, LongType

load_dotenv('../.env')

os.environ['HADOOP_HOME'] = "C:/hadoop"
sys.path.append("C:/hadoop/bin")

In [3]:
def ensure_bucket_exists(bucket_name, region):
    s3_client = boto3.client(
        's3',
        aws_access_key_id=os.getenv('AWS_ACCESS_KEY'),
        aws_secret_access_key=os.getenv('AWS_SECRET_KEY'),
        region_name=region
    )

    try:
        s3_client.head_bucket(Bucket=bucket_name)
        print(f'Bucket {bucket_name} already exists.')
    except ClientError as e:
        error_code = e.response['Error']['Code']

        if error_code == '404':
            print(f'Bucket {bucket_name} not found. Creating...')
            location = {'LocationConstraint': region}
            s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration=location)
            print(f'Bucket {bucket_name} created')
        else:
            print(f"Can't verify bucket! {e}")

In [4]:
packages = [
    'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0',
    'org.apache.hadoop:hadoop-aws:3.3.4',
    'com.amazonaws:aws-java-sdk-bundle:1.12.262'
]

spark = SparkSession.builder \
    .appName("BinanceStreaming") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", ','.join(packages)) \
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY')) \
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_KEY')) \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Py4JJavaError: An error occurred while calling None.org.apache.spark.sql.classic.SparkSession.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:75)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:53)
java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:502)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:486)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(Constru

In [ ]:
raw_df = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', 'kafka-1:9092') \
    .option('subscribe', 'trades.normalized') \
    .option('startingOffsets', 'latest') \
    .load()

In [ ]:
schema = StructType([
    StructField('event', StringType(), True),
    StructField('symbol', StringType(), True),
    StructField('trade_id', StringType(), True),
    StructField('price', DecimalType(18, 8), True),
    StructField('qty', DecimalType(18, 8), True),
    StructField('trade_time', LongType(), True),
    StructField('is_maker', BooleanType(), True)
])

In [ ]:
df = raw_df.select(from_json(col('value').cast('string'), schema).alias('data'))
df = df.select('data.*')

In [ ]:
df = df.withColumn(
    'trade_time_readable', 
    (from_unixtime(col('trade_time') / 1000)).cast('timestamp')
)

In [ ]:
df = df.dropna()

In [ ]:
# bucket_name = 'binance-data-lucas-2026'
# bucket_region = 'sa-east-1'

# ensure_bucket_exists(bucket_name, bucket_region)

# query_s3 = df.writeStream \
#     .format('parquet') \
#     .option('path', f's3a://{bucket_name}/data/normalized/') \
#     .option('checkpointLocation', f's3a://{bucket_name}/checkpoints/') \
#     .outputMode('append') \
#     .start()

# query_s3.awaitTermination()

Bucket binance-data-lucas-2026 already exists.


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
query = df.writeStream \
    .queryName('binance_data') \
    .format('memory') \
    .outputMode('append') \
    .option('startingOffset', 'earliest') \
    .start()

In [ ]:
from IPython.display import clear_output
from time import sleep

while True:
    clear_output(wait=True)
    spark.sql('SELECT * FROM binance_data ORDER BY trade_time DESC LIMIT 10').show(truncate=False)
    sleep(5)

In [ ]:
spark.stop()